# SplitSnap 1: fine-tuning Donut on Kaggle

Reproduces the training run behind the model report: `naver-clova-ix/donut-base` fully fine-tuned on CORD, SROIE and generated Indian-style bills.

**Kaggle settings.** Accelerator: GPU T4 x1. Internet: on. Add the input dataset `urbikn/sroie-datasetv2` (SROIE 2019). CORD is downloaded from Hugging Face by the code.

**Time, measured on a T4:** data preparation about 5 min; training 0.42 s per sample, about 14 min per epoch; 8 epochs per session is about 2.2 h and the 16 epochs take two sessions (126 and 119 min of script time). Kaggle sessions are time-limited, which is why the run is split in two stages that share one 16-epoch learning-rate schedule: stage 2 resumes from the optimiser state saved by stage 1.

**What this notebook is.** It runs the same commands as `kaggle/run_kaggle.py`, the script through which the real run was executed, in the same order and with the same flags. The notebook form itself has not been run end to end on Kaggle; the script has.

Run the cells top to bottom. For stage 2, add the stage 1 notebook output as an input and set `STAGE = 2`.

In [ ]:
import glob, json, os, shutil, subprocess, time

STAGE = 1                       # 1: epochs 1-8, starts from donut-base.   2: epochs 9-16, resumes from stage 1
PREVIOUS_OUTPUT = ""            # stage 2 only: folder of the stage 1 output that contains runs/run1 (find it under /kaggle/input)
EPOCHS, EPOCHS_PER_STAGE = 16, 8
REPO = "https://github.com/atrijopal/naala_ml_tharki_ap.git"
PROJ, OUT = "/kaggle/working/proj", "/kaggle/working"
ENV = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "TOKENIZERS_PARALLELISM": "false"}
T0 = time.time()

def sh(cmd, env=None, check=True):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True, cwd=PROJ, env={**os.environ, **(env or {})})
    if check and r.returncode:
        raise SystemExit(f"FAILED ({r.returncode}): {cmd}")
    return r.returncode

## 1. Code and environment

In [ ]:
shutil.rmtree(PROJ, ignore_errors=True)
subprocess.run(f"git clone -q {REPO} {PROJ}", shell=True, check=True)
os.makedirs(f"{PROJ}/data", exist_ok=True)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; python -V")
sh('pip install -q "transformers>=4.44,<5" datasets accelerate sentencepiece zss nltk scikit-learn bitsandbytes')
sh("python -c \"import torch,transformers;print('torch',torch.__version__,'transformers',transformers.__version__)\"")

## 2. Data

* **CORD** (Indonesian receipts), converted to our schema (items, typed charges, subtotal, grand total).
* **Generated Indian-style bills**: 700 train, 100 validation, 100 test, with fixed seeds. These are whole generated bills, which go beyond light augmentation; the report discloses and separates them.
* **SROIE** (company, date, address, total) as a second task behind its own start token; 60 training scans are held out as validation.
* `build_manifest` joins the sources, assigns the splits and removes duplicate images across splits.

In [ ]:
sh("python -m splitsnap.convert_cord")
for split, n, seed in (("train", 700, 1000), ("val", 100, 2000), ("test", 100, 3000)):
    sh(f"python -m splitsnap.synth --n {n} --out data/synth --seed {seed} --split {split}")
sh("python -m splitsnap.convert_sroie --out data/sroie_manifest.jsonl --val_n 60")      # finds SROIE2019 under /kaggle/input
sh("python -m splitsnap.build_manifest data/cord/manifest.jsonl data/synth/manifest_train.jsonl "
   "data/synth/manifest_val.jsonl data/synth/manifest_test.jsonl data/sroie_manifest.jsonl")

## 3. Overfit gate (stage 1 only)

Trains on 24 samples without augmentation and requires that **generation** reproduces them (selection score at least 0.3; a model that writes nothing scores about 0.15). It fails in about 10 minutes if something is wrong with data, loss or decoding, before the long run starts.

In [ ]:
FLAGS = "--height 1280 --width 960 --bs 1 --accum 8 --lr 3e-5 --val_every 2 --repeat real=3 --workers 3 --max_length 512"
if STAGE == 1:
    GATE = FLAGS.replace("--accum 8", "--accum 2").replace("--lr 3e-5", "--lr 5e-5").replace("--val_every 2", "--val_every 15")
    sh(f"python -m splitsnap.train --manifest data/manifest.jsonl --out {OUT}/gate {GATE} --epochs 15 --max_train 24 "
       f"--val_n 24 --val_on_train --no_aug --no_save --abort_if_val_below 0.3", env=ENV)
    shutil.rmtree(f"{OUT}/gate", ignore_errors=True)

## 4. Training

Full fine-tune, fp16 mixed precision, 1280x960 input, batch 1 with 8-step gradient accumulation (effective batch 8), AdamW with learning rate 3e-5 and a cosine schedule over 16 epochs. Checkpoints are written atomically; the run stops by itself on a NaN loss, a hang or repeated out-of-memory errors. Validation runs every second epoch, and `best/` is chosen by the mean of the CORD and SROIE validation scores.

In [ ]:
resume = ""
if STAGE == 2:
    shutil.copytree(f"{PREVIOUS_OUTPUT}/runs/run1", f"{OUT}/runs/run1", dirs_exist_ok=True)   # contains last/train_state.pt
    resume = "--resume"
rc = sh(f"python -m splitsnap.train --manifest data/manifest.jsonl --out {OUT}/runs/run1 {FLAGS} --epochs {EPOCHS} "
        f"--val_n 60 --max_hours 6.5 {resume} --chunk_epochs {EPOCHS_PER_STAGE}", env=ENV, check=False)
status = f"{OUT}/runs/run1/STATUS.json"
print("exit code", rc, "| status:", open(status).read() if os.path.exists(status) else "(none)")
print(f"{(time.time() - T0) / 60:.0f} min elapsed")

## 5. Curves and metrics

In [ ]:
sh(f"python -m splitsnap.plot_curves --metrics {OUT}/runs/run1/metrics.jsonl --out {OUT}/curves.png", check=False)
from IPython.display import Image, display
if os.path.exists(f"{OUT}/curves.png"):
    display(Image(f"{OUT}/curves.png"))

## 6. Held-out evaluation and export (after stage 2 has completed)

Scores the `best/` checkpoint on the test split of each source (CORD, SROIE, generated), with real fp16 latency, and on validation bills with per-field confidences (used for the calibration). Then exports a half-precision copy (about 411 MB) for the app.

In [ ]:
best = f"{OUT}/runs/run1/best"
if STAGE == 2 and os.path.exists(f"{best}/config.json"):
    sh(f"python -m splitsnap.evaluate --ckpt {best} --manifest data/manifest.jsonl --split test --max_length 512 --out {OUT}/eval_test.json", env=ENV)
    sh(f"python -m splitsnap.evaluate --ckpt {best} --manifest data/manifest.jsonl --split val --sources cord,synth --conf --max_length 512 --out {OUT}/eval_val_conf.json", env=ENV)
    sh(f"python scripts/export_fp16.py --ckpt {best} --out {OUT}/best_fp16")
    ts = f"{OUT}/runs/run1/last/train_state.pt"                  # optimiser state is only needed to resume
    if os.path.exists(ts):
        os.remove(ts)
else:
    print("Evaluation runs after stage 2 (STAGE = 2) has produced runs/run1/best.")

Download `best_fp16/` from the notebook output and place it in `ckpt/best_fp16/` of the repository to run the app (see the README).